# H-mode global confinement $\tau_E$ scaling
Reproduces the basic engineering-variable results of Verdoolaege *et al.* 2021, Nucl. Fusion **61** 076006 (DB5.2.3) from the IMAS-migrated H-mode database.

In [ ]:
import numpy as np

import _simdb_common as sc

db = sc.get_db()
sims = sc.query_dataset(db, "hmode")
print(f"{len(sims)} shots in hmode")

## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_t$ | `summary/global_quantities/b0/value` | T |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $P_{l,th}$ | `summary/global_quantities/power_loss/value` | W |
| $R_{geo}$ | `summary/global_quantities/r0/value` | m |
| $V$ | `summary/global_quantities/volume/value` | m^3 |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $M_{eff}$ | `summary/volume_average/meff_hydrogenic/value` | AMU |
| $\delta$ | `equilibrium/time_slice(0)/boundary/triangularity` | — |
| TOK | `summary/machine` | — |
| PHASE | `temporary/constant_string0d` (by `identifier.name`) | — |
| SELDB5 | `temporary/constant_integer0d` (by `identifier.name`) | — |

Derived (paper definitions): $\kappa_a = V/(2\pi R_{geo}\,\pi a^2)$, $\epsilon = a/R_{geo}$.

In [ ]:
rows = []
for i, sim in enumerate(sims, start=1):
    if i % 100 == 0 or i == len(sims):
        print(f"\r  {i}/{len(sims)} shots ({sim.alias})".ljust(80), end="", flush=True)

    md = sim.meta_dict()
    machine = md.get("machine", "")
    seldb5 = sc.temp(md, "SELDB5", n=0)
    n = len(seldb5)
    if n == 0:
        continue
    tau   = sc.path(md, "global_quantities", "tau_energy", "value", n=n)
    ip    = sc.path(md, "global_quantities", "ip", "value", n=n)
    bt    = np.abs(sc.path(md, "global_quantities", "b0", "value", n=n))
    nel   = sc.path(md, "line_average", "n_e", "value", n=n)
    plth  = sc.path(md, "global_quantities", "power_loss", "value", n=n)
    rgeo  = sc.path(md, "boundary", "geometric_axis_r", "value", n=n)
    vol   = sc.path(md, "global_quantities", "volume", "value", n=n)
    amin  = sc.path(md, "boundary", "minor_radius", "value", n=n)
    meff  = sc.path(md, "volume_average", "meff_hydrogenic", "value", n=n)
    delta = sc.temp(md, "DELTA", n=n)
    phase_raw = md.get("db_variable", {}).get("PHASE")

    n_slices = min(len(a) for a in (seldb5, tau, ip, bt, nel, plth, rgeo, vol, amin, meff, delta))
    if n_slices == 0:
        continue
    for j in range(n_slices):
        phase = phase_raw[j] if phase_raw is not None and j < len(phase_raw) else ""
        rows.append((machine, tau[j], ip[j], bt[j], nel[j], plth[j], rgeo[j], vol[j], amin[j],
                     meff[j], delta[j], phase, seldb5[j]))
print()

dtype = [("tok", "U16"), ("tau", float), ("ip", float), ("bt", float), ("nel", float),
         ("plth", float), ("rgeo", float), ("vol", float), ("amin", float), ("meff", float),
         ("delta", float), ("phase", object), ("seldb5", float)]
data = np.array(rows, dtype=dtype)

N      = len(data)
TAU    = data["tau"]
IP     = data["ip"]
BT     = data["bt"]
NEL    = data["nel"]
PLTH   = data["plth"]
RGEO   = data["rgeo"]
VOL    = data["vol"]
AMIN   = data["amin"]
MEFF   = data["meff"]
DELTA  = data["delta"]
TOK    = data["tok"]
PHASE  = data["phase"]
SELDB5 = data["seldb5"]

print("Done.")

In [3]:
# Units and derived variables (formula in paper)
tau_s    = TAU                                 # [s]
ip_ma    = np.abs(IP) / 1e6                     # [MA]
Bt_T     = np.abs(BT)                           # [T]
ne_19    = NEL / 1e19                           # [10^19 m^-3]
Ploss_MW = PLTH / 1e6                           # [MW]
kappa_a  = VOL / (2.0 * np.pi * RGEO * np.pi * AMIN**2)   # paper kappa_a
eps      = AMIN / RGEO                          # inverse aspect ratio
one_delta = 1.0 + DELTA                         # 1 + delta

# Subset selection: STD5 standard set, ELMy H-mode
PHASE_str = PHASE.astype(str)
std5 = (SELDB5 == 1)
elmy = np.char.startswith(PHASE_str, "HG") | np.char.startswith(PHASE_str, "HS")

# Check if theres a difference?
print(f"Is std5 == elmy?: {np.array_equal(std5, elmy)}")

if not np.array_equal(std5,elmy):
    print(f"No, they differ in: {np.sum(std5 != elmy)} places")
    

regressors = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF]
finite = np.all([np.isfinite(p) for p in regressors], axis=0)
positive = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
sel = std5 & elmy & finite & positive

print(f"Total pulses     : {N}")
print(f"SELDB5 == 1      : {np.sum(std5)}")
print(f"ELMy H (HG/HS)   : {np.sum(elmy)}")
print(f"std5 + ELMy + valid: {np.sum(sel)}")
print("\nper machine (ELMy, valid):")
for tok in np.unique(TOK[sel]):
    print(f"  {tok:10s}: {np.sum((TOK == tok) & sel)}")

Is std5 == elmy?: False
No, they differ in: 2560 places
Total pulses     : 14153
SELDB5 == 1      : 7568
ELMy H (HG/HS)   : 7520
std5 + ELMy + valid: 6250

per machine (ELMy, valid):
  ASDEX     : 431
  AUG       : 2133
  CMOD      : 45
  COMPASS   : 16
  D3D       : 388
  JET       : 2652
  JFT2M     : 70
  JT60U     : 100
  MAST      : 43
  NSTX      : 185
  PBXM      : 59
  PDX       : 97
  START     : 8
  TCV       : 11
  TDEV      : 10
  TFTR      : 2


## Table 2 — engineering-variable ranges (STD5 ELMy H)
Compare with paper Table 9 (DB5.2.3-STD5 ELMy H). 

In [4]:
cols = {
    "tau_E,th [s]": tau_s[sel],
    "Ip [MA]":      ip_ma[sel],
    "Bt [T]":       Bt_T[sel],
    "ne [1e19]":    ne_19[sel],
    "Pl,th [MW]":   Ploss_MW[sel],
    "Rgeo [m]":     RGEO[sel],
    "1+delta":      one_delta[sel],
    "kappa_a":      kappa_a[sel],
    "eps":          eps[sel],
    "Meff":         MEFF[sel],
}
print(f"{'variable':14s} {'min':>9s} {'max':>9s} {'mean':>9s} {'median':>9s} {'std':>9s}")
for name, x in cols.items():
    print(f"{name:14s} {np.min(x):9.4g} {np.max(x):9.4g} {np.mean(x):9.4g} {np.median(x):9.4g} {np.std(x):9.4g}")

variable             min       max      mean    median       std
tau_E,th [s]    0.002236     1.321    0.1773     0.126    0.1475
Ip [MA]           0.1593     5.134     1.387     1.009    0.8099
Bt [T]            0.2613     5.821     2.141      2.19    0.6652
ne [1e19]          1.166      42.6     5.943     5.499     3.196
Pl,th [MW]        0.1464     32.67     7.996     6.611     5.347
Rgeo [m]          0.2804       3.4     2.161     1.677    0.7056
1+delta              nan       nan       nan       nan       nan
kappa_a           0.9308     2.389     1.531     1.562    0.1971
eps               0.1548    0.7831    0.3225    0.3147   0.08269
Meff                   1      3.89     1.916         2    0.2863


## Engineering scaling (OLS in log space)
Power law $\tau_{E,th} = \alpha_0\, I_p^{\alpha_I} B_t^{\alpha_B} \bar n_e^{\alpha_n} P_{l,th}^{\alpha_P} R_{geo}^{\alpha_R} \kappa_a^{\alpha_\kappa} \epsilon^{\alpha_\epsilon} M_{eff}^{\alpha_M}$ (paper eq. 2), fitted by ordinary least squares on $\ln$-transformed data.

In [5]:
# Design matrix in log space (Ip in MA, ne in 1e19, Pl,th in MW — absorbed into intercept)
y = np.log(tau_s[sel])
regressors = {
    "ln Ip":   np.log(ip_ma[sel]),
    "ln Bt":   np.log(Bt_T[sel]),
    "ln ne":   np.log(ne_19[sel]),
    "ln Plth": np.log(Ploss_MW[sel]),
    "ln Rgeo": np.log(RGEO[sel]),
    "ln kapa": np.log(kappa_a[sel]),
    "ln eps":  np.log(eps[sel]),
    "ln Meff": np.log(MEFF[sel]),
}

def paper_weights(tok_arr):
    w = np.ones(len(tok_arr))
    for tok in np.unique(tok_arr):
        m = tok_arr == tok
        w[m] = 1.0 / (2.0 + np.sqrt(m.sum() / 4.0))
    return w

X = np.column_stack([np.ones_like(y)] + list(regressors.values()))
w  = paper_weights(TOK[sel])
sw = np.sqrt(w)
coef, *_ = np.linalg.lstsq(X * sw[:,None], y * sw, rcond=None)

names = ["ln alpha0"] + list(regressors.keys())
# IPB98(y,2) reference exponents (Table 7): aI,aB,an,aP,aR,akappa,aeps,aM
ipb98 = {"ln Ip":0.93, "ln Bt":0.15, "ln ne":0.41, "ln Plth":-0.69,
         "ln Rgeo":1.97, "ln kapa":0.78, "ln eps":0.58, "ln Meff":0.19}

print(f"{'param':10s} {'this OLS':>10s} {'IPB98(y,2)':>12s}")
print(f"{'alpha0':10s} {np.exp(coef[0]):>10.4g} {0.0562:>12.4g}")
for nm, c in zip(names[1:], coef[1:]):
    ref = ipb98.get(nm, np.nan)
    print(f"{nm:10s} {c:>10.3f} {ref:>12.3f}")

resid = y - X @ coef
rmse = np.sqrt(np.mean(resid**2))
r2 = 1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2)
print(f"\nN = {sel.sum()}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}")

param        this OLS   IPB98(y,2)
alpha0        0.07546       0.0562
ln Ip           0.908        0.930
ln Bt           0.195        0.150
ln ne           0.255        0.410
ln Plth        -0.616       -0.690
ln Rgeo         1.711        1.970
ln kapa         0.406        0.780
ln eps          0.449        0.580
ln Meff         0.156        0.190

N = 6250   RMSE(log) = 0.209   R^2 = 0.937


## Table 8 — per-machine OLS scalings (all H-modes in STD5)
OLS on log-transformed data. 

In [7]:
from numpy.linalg import lstsq

def ols_fit(y_log, X):
    """OLS in log space; returns (coef, std_err, resid)."""
    coef, _, _, _ = lstsq(X, y_log, rcond=None)
    n, p = X.shape
    resid = y_log - X @ coef
    # unbiased variance of residuals
    sigma2 = np.sum(resid**2) / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return coef, se, resid

def metrics(resid, y_log, p_free):
    """MdAPE (%), RMSE (log), R^2 — all on log scale"""
    n = len(resid)
    mdape = np.median(np.abs(resid / y_log)) * 100
    rmse  = np.sqrt(np.sum(resid**2) / n)              # no df correction, matches paper
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y_log - y_log.mean())**2)
    r2    = 1.0 - ss_res / ss_tot
    return mdape, rmse, r2

# Paper Table 8 uses "all H-modes" in STD5 (not just ELMy).
# SELDB5 == 1 is the STD5 flag; require all regressors tp be finite & positive.

all_h = (SELDB5 == 1)
preds_all = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF, one_delta]
finite_all = np.all([np.isfinite(p) | np.isnan(p) for p in preds_all], axis=0)
# positivity for the mandatory variables only
pos_all = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
mask = all_h & pos_all & np.isfinite(tau_s)

# log-arrays (NaN-safe; we apply device mask before building X)
LOG = {
    "Ip":    np.log(np.where(ip_ma    > 0, ip_ma,    np.nan)),
    "Bt":    np.log(np.where(Bt_T     > 0, Bt_T,     np.nan)),
    "ne":    np.log(np.where(ne_19    > 0, ne_19,    np.nan)),
    "Plth":  np.log(np.where(Ploss_MW > 0, Ploss_MW, np.nan)),
    "Rgeo":  np.log(np.where(RGEO     > 0, RGEO,     np.nan)),
    "kappa":  np.log(np.where(kappa_a  > 0, kappa_a,  np.nan)),
    "eps":   np.log(np.where(eps      > 0, eps,       np.nan)),
    "Meff":  np.log(np.where(MEFF     > 0, MEFF,     np.nan)),
    "1+d":   np.log(np.where(one_delta > 0, one_delta, np.nan)),
}
LOG_TAU = np.log(np.where(tau_s > 0, tau_s, np.nan))


DEVICE_SPECS = {
    # TOK  : (paper_label,  [regressors])
    "ASDEX"  : ("ASDEX",       ["Ip", "Bt", "ne", "Plth", "Meff"]),
    "D3D"    : ("DIII-D",      ["Ip", "Bt", "ne", "Plth", "1+d", "Meff"]),
    "JT60U"  : ("JT-60U",      ["Ip", "Bt", "ne", "Plth"]),
    "NSTX"   : ("NSTX",        ["Ip", "Bt", "ne", "Plth", "kappa"]),
    "PDX"    : ("PDX",         ["Ip", "Bt", "ne", "Plth"]),
    "CMOD"   : ("Alcator C-Mod", ["Ip",       "ne", "Plth"]),
    "JFT2M"  : ("JFT-2M",      ["Ip",       "ne", "Plth", "Meff"]),
    "MAST"   : ("MAST",        ["Ip",       "ne", "Plth"]),
    "PBXM"   : ("PBX-M",       ["Ip",       "ne", "Plth"])
    }

# paper Table 8 exponent columns (in order of predictors listed above)
COL_ORDER = ["Ip", "Bt", "ne", "Plth", "1+d", "kappa", "Meff"]
COL_LABELS = [r"$\alpha_I$", r"$\alpha_B$", r"$\alpha_n$", r"$\alpha_P$", 
              r"$\alpha_{(1+\delta)}$", r"$\kappa$", r"$\alpha_M$"]

header = (f"{'Device':<14s}  {'n_obs':>5s}  "
       + "   ".join(f"{c:>9s}" for c in COL_LABELS)
       + f"  {'MdAPE%':>7s}  {'RMSE':>6s}  {'R^2':>5s}")
print(header)
print("-" * len(header))

results = {}
for tok, (label, pkeys) in DEVICE_SPECS.items():
    m = mask & (TOK == tok)
    # require all regressors for this device to be finite
    log_preds = [LOG[k] for k in pkeys]
    finite_dev = np.all([np.isfinite(lp) for lp in log_preds], axis=0)
    finite_dev &= np.isfinite(LOG_TAU)
    m = m & finite_dev
    n = int(m.sum())
    if n < len(pkeys) + 2:
        print(f"{label:<14s} {n:>5d}  (too few points)")
        continue

    y = LOG_TAU[m]
    cols_X = [np.ones(n)] + [LOG[k][m] for k in pkeys]
    X = np.column_stack(cols_X)
    coef, se, resid = ols_fit(y, X)
    mdape, rmse, r2 = metrics(resid, y, len(pkeys))

    # build coefficient dict keyed by predictor name
    coef_dict = {k: (coef[i+1], se[i+1]) for i, k in enumerate(pkeys)}

    # format row: show value pm standard error (se) for each column in COL_ORDER, blank if not fitted
    row = f"{label:<14s} {n:>5d}  "
    for col in COL_ORDER:
        if col in coef_dict:
            v, s = coef_dict[col]
            row += f"  {v:+5.3f}pm{s:5.3f}"
        else:
            row += f"  {'-------------':>9s}"
    row += f"  {mdape:7.1f}  {rmse:6.2f}  {r2:5.2f}"
    print(row)
    results[tok] = {"n": n, "coef": coef_dict, "mdape": mdape, "rmse": rmse, "r2": r2}


Device          n_obs  $\alpha_I$   $\alpha_B$   $\alpha_n$   $\alpha_P$   $\alpha_{(1+\delta)}$    $\kappa$   $\alpha_M$   MdAPE%    RMSE    R^2
-------------------------------------------------------------------------------------------------------------------------------------------------
ASDEX            575    +0.754pm0.043  +0.220pm0.060  +0.678pm0.032  -0.649pm0.019  -------------  -------------  +0.049pm0.046      3.0    0.13   0.80
DIII-D           502    +1.101pm0.047  +0.100pm0.058  +0.094pm0.035  -0.633pm0.021  +0.546pm0.085  -------------  +0.429pm0.057      5.7    0.18   0.79
JT-60U           100    +0.699pm0.144  +0.517pm0.191  -0.175pm0.083  -0.336pm0.063  -------------  -------------  -------------      5.0    0.13   0.82
NSTX             230    +0.215pm0.078  +1.330pm0.091  +0.617pm0.079  -0.703pm0.044  -------------  +0.681pm0.170  -------------      2.5    0.15   0.72
PDX              119    +1.160pm0.198  +0.516pm0.196  +0.417pm0.099  -0.872pm0.094  -------------  -

All non-B_t scaling match table 8 coefficients, RMSE, $R^2$, and MdAPE *exactly*, except JFT-2M, which has an additional pulse, with coefficients consistent with an extra data point.

ASDEX: B, I, M coefficients differ significantly; n, P match exactly.